[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burzynski-lab/OU_PHYS2222_ComputationalPhysics/blob/main/LectureNotes/11_Schrodinger_Eigenvalues.ipynb)

# The Schrödinger Equation as an Eigenvalue Problem

**PHYS 2222: Computational Physics** · University of Oklahoma · Spring 2027 · Week 8

> **Before you start:** in Colab, choose **File → Save a copy in Drive** so your changes are kept.

## Learning goals

By the end of this notebook you should be able to:

- explain, in words and with a simulation, why bound quantum states exist only at certain energies;
- solve the time-independent Schrödinger equation with the **shooting method**: integrate it as an ODE and use root finding (`brentq`) to find the energies;
- normalize a numerical wavefunction by integration;
- use NumPy for basic **linear algebra**: matrices, the `@` operator, `np.linalg.solve`, and **eigenvalues and eigenvectors** with `np.linalg.eigh`;
- turn the Schrödinger equation into a **matrix eigenvalue problem** with finite differences, and check how the answers converge as the grid is refined;
- apply both methods to the infinite square well, the harmonic oscillator, the finite square well, and a potential with no analytic solution.

This notebook connects to Project 2. It teaches the methods; the project asks you to push them further.

## 1. Physics recap: the time-independent Schrödinger equation

In quantum mechanics, a particle in one dimension is described by a **wavefunction** $\psi(x)$. Its square, $|\psi(x)|^2$, is the **probability density**: the probability of finding the particle between $x$ and $x + dx$ is $|\psi(x)|^2\,dx$. Because the particle must be somewhere, a physical wavefunction is **normalized**:

$$
\int_{-\infty}^{\infty} |\psi(x)|^2\, dx = 1 .
$$

A state with definite energy $E$ in a potential energy $V(x)$ satisfies the **time-independent Schrödinger equation** (TISE):

$$
-\frac{\hbar^2}{2m}\,\frac{d^2\psi}{dx^2} + V(x)\,\psi(x) = E\,\psi(x).
$$

The first term is the kinetic energy (it measures how sharply $\psi$ curves), the second the potential energy. A **bound state** is one where the particle is trapped, so $\psi \to 0$ far away. The central fact of quantum mechanics that we will reproduce numerically: **bound states exist only for special values of $E$**, the allowed energy levels. Two results you met in Modern Physics, which we will use as checks:

- **Infinite square well** of width $L$ ($V = 0$ for $0 < x < L$, walls at both ends where $\psi = 0$):
$$E_n = \frac{n^2\pi^2\hbar^2}{2mL^2}, \qquad \psi_n(x) = \sqrt{\tfrac{2}{L}}\,\sin\frac{n\pi x}{L}, \qquad n = 1, 2, 3, \ldots$$
- **Harmonic oscillator** $V = \tfrac12 k x^2$ with $\omega = \sqrt{k/m}$:
$$E_n = \left(n + \tfrac12\right)\hbar\omega, \qquad n = 0, 1, 2, \ldots$$

Rearranged, the TISE is a second-order ODE, exactly the kind you solved in the last notebook (with $x$ playing the role of time):

$$
\frac{d^2\psi}{dx^2} = \frac{2m}{\hbar^2}\,\big(V(x) - E\big)\,\psi(x).
$$

### Units

For an electron in atomic-scale wells, SI units give awkward numbers (lengths near $10^{-9}$ m, energies near $10^{-19}$ J). We will measure **energy in eV** and **length in nm**. The only combination of constants that appears is

$$
C \equiv \frac{\hbar^2}{2m_e} \approx 0.0381\ \text{eV nm}^2 ,
$$

so the TISE becomes $\psi'' = (V - E)\,\psi / C$. Let's compute $C$ from `scipy.constants` rather than trusting the number.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.constants as const
from scipy.integrate import solve_ivp, simpson
from scipy.optimize import brentq

C = const.hbar**2 / (2 * const.m_e) / const.e / 1e-18   # J m^2 -> eV nm^2
print(f"hbar^2 / 2m_e = {C:.5f} eV nm^2")

Dividing by `const.e` converts joules to eV (1 eV $= e \times 1$ V), and dividing by `1e-18` converts m² to nm².

## 2. The shooting method: why only certain energies work

### The idea

Take the infinite square well with $L = 1$ nm. Inside, $V = 0$ and the TISE is $\psi'' = -E\psi/C$. The walls force $\psi(0) = 0$ and $\psi(L) = 0$. That is a **boundary value problem** (conditions at two ends), not an initial value problem. The shooting method turns it into an IVP:

1. Start at the left wall with $\psi(0) = 0$ and some slope, say $\psi'(0) = 1$. (The slope only sets the overall scale of $\psi$, which we fix later by normalizing.)
2. **Guess** an energy $E$ and integrate the ODE across the well, like firing a cannon at a target.
3. Look at where you land: $\psi(L)$. For most energies it is not zero, so that $E$ is not allowed.
4. Adjust $E$ until $\psi(L) = 0$. That is a **root-finding** problem in $E$, which `brentq` solves.

As in the last notebook, we write the second-order ODE as a first-order system with state vector $(\psi, \psi')$.

In [ ]:
def tise(x, s, E, V):
    """Right-hand side of the TISE as a first-order system. s = [psi, dpsi/dx].
    V is a function V(x) returning the potential in eV."""
    psi, dpsi = s
    return [dpsi, (V(x) - E) * psi / C]

L = 1.0                                  # well width (nm)
def V_box(x):
    return 0.0                           # V = 0 inside the infinite well

def psi_at_right_wall(E):
    """Shoot from x = 0 with psi = 0, psi' = 1; return psi(L)."""
    sol = solve_ivp(tise, (0, L), [0.0, 1.0], args=(E, V_box), rtol=1e-10, atol=1e-12)
    return sol.y[0, -1]

Before finding roots, **look** at what happens for a few trial energies near the expected ground state, $E_1 = \pi^2 C/L^2 \approx 0.376$ eV.

In [ ]:
E1_exact = np.pi**2 * C / L**2
x = np.linspace(0, L, 200)
fig, ax = plt.subplots()
for E in [0.25, E1_exact, 0.5]:
    sol = solve_ivp(tise, (0, L), [0.0, 1.0], args=(E, V_box), t_eval=x, rtol=1e-10, atol=1e-12)
    ax.plot(x, sol.y[0], label=f"E = {E:.3f} eV")
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("x (nm)")
ax.set_ylabel(r"$\psi(x)$ (unnormalized)")
ax.set_title("Shooting across an infinite well")
ax.legend()
plt.show()

Too little energy: $\psi$ doesn't curve enough and misses zero at the wall. Too much: it curves too much and crosses zero early. Only at the right energy does it land exactly on $\psi(L) = 0$. This is quantization, seen as a simulation: the wall condition can only be met at special energies.

(Note that `solve_ivp` calls its independent variable `t`, so `t_eval=x` simply means "report the solution at these $x$ values".)

Now scan $\psi(L)$ over a range of energies. Every zero crossing is an allowed energy.

In [ ]:
E_scan = np.linspace(0.05, 10, 200)
psiL = np.array([psi_at_right_wall(E) for E in E_scan])

fig, ax = plt.subplots()
ax.plot(E_scan, psiL)
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("trial energy E (eV)")
ax.set_ylabel(r"$\psi(L)$")
ax.set_title("Allowed energies are the zeros")
plt.show()

### Finding the energies with `brentq`

From the root-finding notebook: `brentq(f, a, b)` finds a root of `f` between `a` and `b`, provided `f(a)` and `f(b)` have opposite signs. The scan gives us those brackets automatically: wherever two neighboring scan values have opposite signs, there is a root between them. The function below is general: give it any "mismatch" function and a scan grid.

In [ ]:
def find_energies(mismatch, E_grid):
    """Return the roots of mismatch(E), bracketed by sign changes on E_grid."""
    values = np.array([mismatch(E) for E in E_grid])
    roots = []
    for i in range(len(E_grid) - 1):
        if values[i] * values[i + 1] < 0:          # sign change -> a root in between
            roots.append(brentq(mismatch, E_grid[i], E_grid[i + 1], xtol=1e-12))
    return np.array(roots)

E_shoot = find_energies(psi_at_right_wall, E_scan)
for n, E in enumerate(E_shoot, start=1):
    print(f"n = {n}:  shooting E = {E:.6f} eV   exact n^2 E1 = {n**2 * E1_exact:.6f} eV")

`enumerate(..., start=1)` hands you a counter along with each element, here starting at 1 so `n` matches the quantum number.

The shooting energies agree with $n^2\pi^2\hbar^2/(2mL^2)$ to about nine digits, set by the `solve_ivp` tolerances.

### Normalization

The shooting solution has an arbitrary scale (we chose $\psi'(0) = 1$). To make $\int_0^L |\psi|^2 dx = 1$, compute the integral numerically with Simpson's rule and divide $\psi$ by its square root.

In [ ]:
x = np.linspace(0, L, 401)
fig, ax = plt.subplots()
for n, E in enumerate(E_shoot[:3], start=1):
    sol = solve_ivp(tise, (0, L), [0.0, 1.0], args=(E, V_box), t_eval=x, rtol=1e-10, atol=1e-12)
    psi = sol.y[0]
    psi = psi / np.sqrt(simpson(psi**2, x=x))       # normalize
    line, = ax.plot(x, psi, label=f"n = {n} (shooting)")
    ax.plot(x, np.sqrt(2 / L) * np.sin(n * np.pi * x / L), "k:", lw=2)
ax.set_xlabel("x (nm)")
ax.set_ylabel(r"$\psi_n(x)$ (nm$^{-1/2}$)")
ax.set_title("dotted: exact")
ax.legend()
plt.show()

The normalized numerical wavefunctions lie on top of $\sqrt{2/L}\sin(n\pi x/L)$. Note the units: $|\psi|^2$ is a probability *per length*, so $\psi$ has units of nm$^{-1/2}$.

### Your turn 1

The formula says $E_1 \propto 1/L^2$: squeeze an electron into a smaller box and its minimum energy goes up. (This "confinement energy" is why atoms have a size and why quantum dots of different sizes glow in different colors.) Use the shooting method to find the ground-state energy for well widths $L = 0.2, 0.5, 1, 2, 5$ nm. Plot $E_1$ versus $L$ on log-log axes and check that the slope is $-2$. Hint: you will need to adjust the energy scan range for each $L$; the formula tells you roughly where to look.

In [ ]:
# Your code here

## 3. The harmonic oscillator by shooting

**Physics recap.** A particle in the potential $V(x) = \tfrac12 k x^2$ feels a restoring force $F = -kx$, like a mass on a spring. It models any system near a stable equilibrium: atoms in a molecule, atoms in a crystal. Classically it oscillates at $\omega = \sqrt{k/m}$ with any energy; quantum mechanically the energies are $(n + \tfrac12)\hbar\omega$, evenly spaced, and the lowest is not zero.

Let's choose $\hbar\omega = 1$ eV for an electron. Since $\hbar\omega = \hbar\sqrt{k/m} = \sqrt{2Ck}$, that means $k = (\hbar\omega)^2/(2C)$. Then we expect $E = 0.5, 1.5, 2.5, \ldots$ eV.

### Two new difficulties

1. **There are no walls.** $\psi$ should go to zero as $x \to \pm\infty$. In practice we integrate out to some $x_{\max}$ far into the region where $V > E$ (the *classically forbidden* region) and demand $\psi(x_{\max}) \approx 0$.
2. **Where do we start?** Here we use **symmetry**. Since $V(-x) = V(x)$, every bound state is either **even**, $\psi(-x) = \psi(x)$, or **odd**, $\psi(-x) = -\psi(x)$. So we only need $x \ge 0$:
   - even states: start at $x = 0$ with $\psi(0) = 1$, $\psi'(0) = 0$ (flat at the center);
   - odd states: start with $\psi(0) = 0$, $\psi'(0) = 1$ (zero at the center).

In the forbidden region the TISE solutions grow or decay exponentially. For a wrong energy the growing one takes over and $\psi(x_{\max})$ shoots off to $+\infty$ or $-\infty$; as $E$ passes through an allowed value the sign flips. So $\psi(x_{\max})$ is again a good mismatch function.

In [ ]:
hw = 1.0                       # hbar * omega (eV)
k_spring = hw**2 / (2 * C)     # eV / nm^2
x0 = np.sqrt(2 * C / hw)       # oscillator length sqrt(hbar / m omega), nm
print(f"k = {k_spring:.3f} eV/nm^2,  oscillator length x0 = {x0:.4f} nm")

def V_ho(x):
    return 0.5 * k_spring * x**2

x_max = 2.0                    # nm, about 7 oscillator lengths

def psi_at_xmax(E, parity):
    start = [1.0, 0.0] if parity == "even" else [0.0, 1.0]
    sol = solve_ivp(tise, (0, x_max), start, args=(E, V_ho), rtol=1e-10, atol=1e-12)
    return sol.y[0, -1]

`[1.0, 0.0] if parity == "even" else [0.0, 1.0]` is a **conditional expression**: a one-line `if/else` that picks one of two values.

Look at the even solutions for energies just below, at, and just above $0.5$ eV:

In [ ]:
x = np.linspace(0, x_max, 400)
fig, ax = plt.subplots()
for E in [0.45, 0.5, 0.55]:
    sol = solve_ivp(tise, (0, x_max), [1.0, 0.0], args=(E, V_ho), t_eval=x, rtol=1e-10, atol=1e-12)
    ax.plot(x, sol.y[0], label=f"E = {E} eV")
ax.set_ylim(-3, 3)
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("x (nm)")
ax.set_ylabel(r"$\psi(x)$ (even, unnormalized)")
ax.legend()
plt.show()

The tail "wags": below the allowed energy $\psi$ blows up upward, above it blows up downward. Only at $E = 0.5$ eV does it die away. Now find the energies of both parities.

In [ ]:
E_grid = np.linspace(0.1, 6, 60)
E_even = find_energies(lambda E: psi_at_xmax(E, "even"), E_grid)
E_odd = find_energies(lambda E: psi_at_xmax(E, "odd"), E_grid)
E_ho = np.sort(np.concatenate([E_even, E_odd]))
for n, E in enumerate(E_ho):
    print(f"n = {n}:  E = {E:.6f} eV   exact (n + 1/2) hbar omega = {n + 0.5:.1f} eV")

Even states give $n = 0, 2, 4, \ldots$ and odd states $n = 1, 3, 5, \ldots$, and together they reproduce $(n + \tfrac12)\hbar\omega$. Here the cutoff at $x_{\max} = 2$ nm is far enough out not to matter. Higher states extend further from the center, so for them you would have to move $x_{\max}$ out as well.

## 4. Linear algebra in NumPy, from scratch

The second method turns the Schrödinger equation into a problem about **matrices**. You don't need a linear algebra course for this; here is what you need.

### Vectors and matrices

A **vector** is a list of numbers; a **matrix** is a rectangular table of numbers. In NumPy these are 1D and 2D arrays. The most important operation is the **matrix–vector product** $A\mathbf v$: each entry of the result is the dot product of a **row** of $A$ with $\mathbf v$,

$$
\begin{pmatrix} a & b \\ c & d \end{pmatrix}
\begin{pmatrix} v_1 \\ v_2 \end{pmatrix}
=
\begin{pmatrix} a v_1 + b v_2 \\ c v_1 + d v_2 \end{pmatrix}.
$$

So a matrix is a machine that takes in a vector and puts out another vector. In Python, the matrix product is written with the **`@` operator**. Be careful: `*` multiplies arrays **element by element**, which is something different.

In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])
v = np.array([1.0, 2.0])

print("A @ v =", A @ v, "   (row 1: 2*1 + 1*2 = 4, row 2: 1*1 + 3*2 = 7)")
print("A * v =\n", A * v, "\n   (element-by-element with broadcasting: NOT a matrix product)")
print("A @ A =\n", A @ A)
print("shape of A:", A.shape, "  transpose A.T =\n", A.T)

Useful constructors: `np.eye(n)` is the $n\times n$ **identity matrix** (ones on the diagonal; $I\mathbf v = \mathbf v$), `np.zeros((n, n))`, and `np.diag(d, k)`, which builds a matrix with the 1D array `d` along a diagonal. `k = 0` is the main diagonal; `k = 1` is the one just above it, `k = -1` just below.

In [ ]:
print(np.eye(3))
print(np.diag([1.0, 2.0, 3.0]) + np.diag([7.0, 8.0], k=1))

### Solving linear equations: `np.linalg.solve`

Many problems reduce to several linear equations in several unknowns, which can be written $A\mathbf x = \mathbf b$. Example: a ball is thrown upward and its height is measured at three times. In free fall $z(t) = z_0 + v_0 t - \tfrac12 g t^2$, so each measurement gives one linear equation in the unknowns $(z_0, v_0, g)$:

$$
\begin{pmatrix} 1 & t_1 & -t_1^2/2 \\ 1 & t_2 & -t_2^2/2 \\ 1 & t_3 & -t_3^2/2 \end{pmatrix}
\begin{pmatrix} z_0 \\ v_0 \\ g \end{pmatrix}
=
\begin{pmatrix} z_1 \\ z_2 \\ z_3 \end{pmatrix}.
$$

`np.linalg.solve(A, b)` finds $\mathbf x$. (The heights below are made up, generated from $z_0 = 1.5$ m, $v_0 = 12$ m/s, $g = 9.8$ m/s².)

In [ ]:
t_meas = np.array([0.5, 1.0, 2.0])                               # s
z_meas = 1.5 + 12.0 * t_meas - 0.5 * 9.8 * t_meas**2              # m (simulated, exact)
M = np.column_stack([np.ones(3), t_meas, -t_meas**2 / 2])         # columns side by side
z0_fit, v0_fit, g_fit = np.linalg.solve(M, z_meas)
print(f"z0 = {z0_fit:.3f} m, v0 = {v0_fit:.3f} m/s, g = {g_fit:.3f} m/s^2")

`np.column_stack` glues 1D arrays together as the columns of a matrix. (When there are more measurements than unknowns, you need a *least-squares* fit instead; that comes in the fitting notebooks.)

### Eigenvalues and eigenvectors

Usually a matrix changes both the length and the **direction** of a vector. But for special vectors, it only stretches (or shrinks, or flips) them:

$$
A\mathbf v = \lambda \mathbf v .
$$

Such a $\mathbf v$ is an **eigenvector** of $A$ and the stretch factor $\lambda$ is its **eigenvalue** ("eigen" is German for "own" or "characteristic"). An $n\times n$ matrix has $n$ eigenvalues. The picture below draws several unit vectors (gray) and where the matrix $A$ above sends them (blue). The two eigenvectors (red) are the only directions that don't rotate.

In [ ]:
lam, vecs = np.linalg.eigh(A)
fig, ax = plt.subplots(figsize=(5.5, 5.5))
for ang in np.linspace(0, np.pi, 9, endpoint=False):
    u = np.array([np.cos(ang), np.sin(ang)])
    Au = A @ u
    ax.annotate("", xy=u, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="gray"))
    ax.annotate("", xy=Au, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="tab:blue", alpha=0.6))
for i in range(2):
    ax.annotate("", xy=lam[i] * vecs[:, i], xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color="tab:red", lw=2))
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.set_aspect("equal")
ax.set_title("gray: u,  blue: A @ u,  red: eigenvectors times eigenvalue")
plt.show()

`np.linalg.eigh(A)` returns two things:

- `lam`: a 1D array of eigenvalues, **sorted from smallest to largest**;
- `vecs`: a 2D array whose **columns** are the eigenvectors: `vecs[:, i]` goes with `lam[i]`. (A common mistake is to take rows, `vecs[i]`, instead.) Each eigenvector has length 1.

`eigh` is for **symmetric** matrices ($A = A^T$; for complex matrices, *Hermitian*). These always have real eigenvalues and mutually perpendicular eigenvectors. Quantum Hamiltonians are always Hermitian, so `eigh` is the one we want. For a general matrix there is `np.linalg.eig`, which is slower and can return complex numbers in unsorted order.

Always check the defining equation:

In [ ]:
print("eigenvalues:", lam)
for i in range(2):
    v_i = vecs[:, i]
    print(f"A @ v{i} - lam{i} * v{i} =", A @ v_i - lam[i] * v_i)

### A physical eigenvalue problem: coupled oscillators

Two equal masses $m$ sit between two walls, joined wall–spring–mass–spring–mass–spring–wall, all springs with stiffness $k$. With displacements $x_1, x_2$, Newton's law for each mass is

$$
m\ddot x_1 = -2k x_1 + k x_2, \qquad m\ddot x_2 = k x_1 - 2k x_2 ,
\qquad\text{or}\qquad
m\ddot{\mathbf x} = -K\mathbf x,\quad K = \begin{pmatrix} 2k & -k \\ -k & 2k \end{pmatrix}.
$$

A **normal mode** is a motion where both masses oscillate at the same frequency, $\mathbf x(t) = \mathbf v\cos\omega t$. Plugging in gives $K\mathbf v = m\omega^2\,\mathbf v$: an eigenvalue problem! The eigenvectors are the shapes of the normal modes and the eigenvalues give their frequencies.

In [ ]:
k_s, m_s = 1.0, 1.0                     # N/m, kg
K = np.array([[2 * k_s, -k_s],
              [-k_s, 2 * k_s]])
lam, vecs = np.linalg.eigh(K / m_s)
for i in range(2):
    print(f"mode {i}: omega = {np.sqrt(lam[i]):.4f} rad/s, shape (x1, x2) = {vecs[:, i]}")
print("expected: omega = 1 (masses in step) and sqrt(3) = 1.7321 (masses opposite)")

The slow mode has the masses moving together (the middle spring is never stretched); the fast mode has them moving in opposite directions. The overall sign of an eigenvector is arbitrary, so you may see $(-0.707, -0.707)$ instead of $(0.707, 0.707)$.

The quantum connection: the TISE, $\hat H\psi = E\psi$, has the same form. The Hamiltonian $\hat H = -\frac{\hbar^2}{2m}\frac{d^2}{dx^2} + V(x)$ acts on a function and gives back another function; the allowed energies are its eigenvalues and the wavefunctions its eigenvectors. Next we turn $\hat H$ into an actual matrix.

### Your turn 2

Extend the coupled-oscillator example to **three** equal masses between two walls (four equal springs). Write down the $3\times3$ matrix $K$ (each mass is pulled by its two neighbors), find the three normal-mode frequencies and shapes with `np.linalg.eigh`, and verify $K\mathbf v = m\omega^2 \mathbf v$ for each mode. Describe each mode's motion in words. (Exact: $\omega^2 = (2 - \sqrt2)k/m,\ 2k/m,\ (2 + \sqrt2)k/m$.)

In [ ]:
# Your code here

## 5. The matrix (finite-difference) method

### From a differential equation to a matrix

Put the wavefunction on a grid of $N$ points $x_j = x_{\text{left}} + jh$, $j = 1, \ldots, N$, with spacing $h$, and assume hard walls just outside the grid: $\psi_0 = \psi_{N+1} = 0$. From the differentiation notebook, the central-difference second derivative is

$$
\psi''(x_j) \approx \frac{\psi_{j+1} - 2\psi_j + \psi_{j-1}}{h^2}, \qquad \text{error } O(h^2).
$$

Substitute into the TISE, $-C\psi'' + V\psi = E\psi$, at every grid point:

$$
-\frac{C}{h^2}\,\psi_{j-1} + \left(\frac{2C}{h^2} + V_j\right)\psi_j - \frac{C}{h^2}\,\psi_{j+1} = E\,\psi_j .
$$

That is $N$ linear equations, one per grid point, linking each $\psi_j$ to its neighbors. With the vector $\boldsymbol\psi = (\psi_1, \ldots, \psi_N)$ it is exactly $H\boldsymbol\psi = E\boldsymbol\psi$ with the matrix (shown for $N = 4$, $t \equiv C/h^2$)

$$
H = \begin{pmatrix}
2t + V_1 & -t & 0 & 0 \\
-t & 2t + V_2 & -t & 0 \\
0 & -t & 2t + V_3 & -t \\
0 & 0 & -t & 2t + V_4
\end{pmatrix}.
$$

Only three diagonals are nonzero: it is a **tridiagonal** matrix. One call to `eigh` gives *all* $N$ energies and wavefunctions at once, with no guessing and no brackets.

In [ ]:
def hamiltonian(x, V):
    """Finite-difference Hamiltonian (eV) on the evenly spaced interior grid x (nm).
    V is an array of potential values at the grid points. Walls (psi = 0) lie one step outside x."""
    h = x[1] - x[0]
    t = C / h**2
    N = len(x)
    return np.diag(2 * t + V) + np.diag(-t * np.ones(N - 1), k=1) + np.diag(-t * np.ones(N - 1), k=-1)

print(np.round(hamiltonian(np.linspace(0.2, 0.8, 4), np.zeros(4)), 3))

### Infinite square well

For the well $0 < x < L$ with $N$ interior points, $h = L/(N+1)$ and the grid is $x_j = jh$. The walls at $0$ and $L$ are then exactly one step outside the grid.

**Normalizing:** `eigh` returns eigenvectors with $\sum_j |v_j|^2 = 1$. We want $\int|\psi|^2dx \approx \sum_j |\psi_j|^2 h = 1$, so $\psi_j = v_j/\sqrt h$.

In [ ]:
N = 500
h = L / (N + 1)
x = h * np.arange(1, N + 1)          # interior points only
H = hamiltonian(x, np.zeros(N))
E_mat, vecs = np.linalg.eigh(H)
psi_mat = vecs / np.sqrt(h)          # each column is a normalized wavefunction

for n in range(1, 6):
    print(f"n = {n}: matrix E = {E_mat[n - 1]:.6f} eV   exact = {n**2 * E1_exact:.6f} eV")

Good agreement, but not to nine digits like shooting: the finite-difference second derivative has an $O(h^2)$ error. Higher states, which wiggle faster, are less accurate on a fixed grid. Let's measure the convergence, as we did for every method so far.

In [ ]:
Ns = np.array([25, 50, 100, 200, 400, 800])
err1, err5 = [], []
for N in Ns:
    h = L / (N + 1)
    E = np.linalg.eigvalsh(hamiltonian(h * np.arange(1, N + 1), np.zeros(N)))
    err1.append(abs(E[0] / E1_exact - 1))
    err5.append(abs(E[4] / (25 * E1_exact) - 1))
hs = L / (Ns + 1)

fig, ax = plt.subplots()
ax.loglog(hs, err1, "o-", label="n = 1")
ax.loglog(hs, err5, "s-", label="n = 5")
ax.loglog(hs, err1[0] * (hs / hs[0])**2, "k--", label=r"$\propto h^2$")
ax.set_xlabel("grid spacing h (nm)")
ax.set_ylabel("relative error in E")
ax.legend()
plt.show()

`np.linalg.eigvalsh` is `eigh` without the eigenvectors: faster when you only want the energies. Both curves have slope 2: double the number of grid points and the error drops by 4. The $n = 5$ error is about 25 times larger at the same $h$, because the error depends on how many grid points fit into one wavelength.

### A faster tool for tridiagonal matrices

A dense $N\times N$ matrix is mostly zeros here, and `eigh` takes a time that grows like $N^3$. SciPy's `scipy.linalg.eigh_tridiagonal(d, e)` takes only the main diagonal `d` and the off-diagonal `e`, and can compute just the lowest few eigenvalues with `select="i", select_range=(0, n-1)`.

In [ ]:
from scipy.linalg import eigh_tridiagonal
import time

N = 3000
h = L / (N + 1)
d = np.full(N, 2 * C / h**2)               # main diagonal (V = 0 here)
e = np.full(N - 1, -C / h**2)              # off-diagonal

t_start = time.perf_counter()
E_tri = eigh_tridiagonal(d, e, eigvals_only=True, select="i", select_range=(0, 4))
t_tri = time.perf_counter() - t_start

t_start = time.perf_counter()
E_dense = np.linalg.eigvalsh(np.diag(d) + np.diag(e, 1) + np.diag(e, -1))
t_dense = time.perf_counter() - t_start

print("lowest 5 (tridiagonal):", E_tri)
print("lowest 5 (dense eigh) :", E_dense[:5])
print(f"time: tridiagonal {t_tri:.3f} s, dense {t_dense:.3f} s")

`np.full(N, value)` makes an array of `N` copies of `value`. `time.perf_counter()` reads a precise clock, so the difference of two readings is the elapsed time.

## 6. The harmonic oscillator with the matrix method

The matrix method needs walls, but we can simply put them far out, where the wavefunctions we care about are already tiny. Use the same oscillator as in Section 3 ($\hbar\omega = 1$ eV) on $-2 < x < 2$ nm.

In [ ]:
N = 1000
x_left, x_right = -2.0, 2.0
h = (x_right - x_left) / (N + 1)
x = x_left + h * np.arange(1, N + 1)
V = V_ho(x)
E_ho_mat, vecs = np.linalg.eigh(hamiltonian(x, V))
psi_ho = vecs / np.sqrt(h)

for n in range(6):
    print(f"n = {n}: matrix E = {E_ho_mat[n]:.5f} eV   exact = {n + 0.5:.1f} eV")

A standard way to display bound states: draw the potential, and draw each wavefunction **shifted up by its energy**, so each one sits on its energy level. You can see where each state is classically allowed (inside the parabola) and where it leaks into the forbidden region.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(x, V, "k", lw=2, label="V(x)")
scale = 0.3                                   # visual scale for psi (eV per nm^(-1/2))
for n in range(5):
    ax.axhline(E_ho_mat[n], color="gray", lw=0.5, ls=":")
    ax.plot(x, E_ho_mat[n] + scale * psi_ho[:, n], label=f"n = {n}")
ax.set_ylim(0, 5.5)
ax.set_xlim(-1.5, 1.5)
ax.set_xlabel("x (nm)")
ax.set_ylabel("energy (eV)")
ax.set_title(r"Harmonic oscillator, $\hbar\omega$ = 1 eV (wavefunctions offset by energy)")
ax.legend(loc="upper right", fontsize=8)
plt.show()

Notice that state $n$ has $n$ nodes (zero crossings), that the states alternate even and odd, and that each wavefunction extends a little beyond the classical turning points where $V = E$. That leakage is the same physics that makes **tunneling** possible.

### Your turn 3

The eigenvectors of a symmetric matrix are **orthogonal**. For wavefunctions this means $\int \psi_m \psi_n\,dx = 0$ for $m \ne n$, and $1$ for $m = n$. Using `psi_ho`, compute the $5\times 5$ table of overlaps $\int \psi_m\psi_n\,dx \approx \sum_j \psi_m(x_j)\psi_n(x_j)\,h$ for $m, n = 0, \ldots, 4$. Do it in a single line with a matrix product (hint: `psi_ho[:, :5].T @ psi_ho[:, :5] * h`), print it with `np.round(..., 6)`, and explain what you see.

In [ ]:
# Your code here

## 7. The finite square well: comparing with the transcendental equations

**Physics recap.** In a finite well the walls have a finite height $V_0$. We take $V = 0$ for $|x| < a$ (width $2a$) and $V = V_0$ outside. (In the root-finding notebook we put the zero of energy at the *top* of the well instead, $V = -V_0$ inside and $0$ outside. That just shifts every energy by $V_0$: our $E$ here is the same as $E + V_0$ there, the energy above the bottom of the well.) A bound state ($0 < E < V_0$) oscillates inside, $\psi \propto \cos kx$ or $\sin kx$ with $k = \sqrt{E/C}$, and decays exponentially outside, $\psi \propto e^{-\kappa|x|}$ with $\kappa = \sqrt{(V_0 - E)/C}$. Requiring $\psi$ and $\psi'$ to be continuous at $x = a$ gives the transcendental equations you solved in the root-finding notebook:

$$
\text{even: } k\tan(ka) = \kappa, \qquad \text{odd: } -k\cot(ka) = \kappa .
$$

With $z = ka$ and $z_0 = a\sqrt{V_0/C}$ these become $z\tan z = \sqrt{z_0^2 - z^2}$ and $-z\cot z = \sqrt{z_0^2 - z^2}$. To avoid the infinities of $\tan$ and $\cot$, multiply through by $\cos z$ or $\sin z$; the roots stay the same.

We use an electron in a well with $a = 0.5$ nm (width 1 nm) and $V_0 = 5$ eV.

In [ ]:
a_well, V0 = 0.5, 5.0                     # nm, eV
z0 = a_well * np.sqrt(V0 / C)

def even_eq(z):
    return z * np.sin(z) - np.sqrt(z0**2 - z**2) * np.cos(z)

def odd_eq(z):
    return z * np.cos(z) + np.sqrt(z0**2 - z**2) * np.sin(z)

z_grid = np.linspace(1e-6, z0 - 1e-9, 2000)
z_roots = np.sort(np.concatenate([find_energies(even_eq, z_grid), find_energies(odd_eq, z_grid)]))
E_transc = C * (z_roots / a_well)**2      # E = C k^2
print(f"z0 = {z0:.3f}: {len(E_transc)} bound states")
print("transcendental-equation energies (eV):", np.round(E_transc, 5))

We reused `find_energies` as a general "find all roots on a grid" tool, here with $z$ instead of $E$.

Now the matrix method. The grid must extend well beyond the well so that the decaying tails fit before the artificial hard walls.

In [ ]:
def V_finite(x):
    return np.where(np.abs(x) < a_well, 0.0, V0)

N = 1500
x_left, x_right = -2.0, 2.0
h = (x_right - x_left) / (N + 1)
x = x_left + h * np.arange(1, N + 1)
E_fw, vecs = np.linalg.eigh(hamiltonian(x, V_finite(x)))
psi_fw = vecs / np.sqrt(h)

n_bound = np.sum(E_fw < V0)
print("matrix-method energies below V0 (eV):", np.round(E_fw[:n_bound], 5))
print("difference from transcendental      :", np.round(E_fw[:n_bound] - E_transc, 5))

`np.where(condition, a, b)` picks `a` where the condition is true and `b` elsewhere, element by element: an `if/else` for whole arrays. `np.sum(E_fw < V0)` counts the `True` values.

The two completely different methods agree to within about 1 to 12 meV, that is, to better than 0.5%. The agreement is limited mainly by the grid: the potential jumps abruptly at $x = \pm a$, which costs some accuracy on a finite grid.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(x, V_finite(x), "k", lw=2)
for n in range(n_bound):
    ax.axhline(E_fw[n], color="gray", lw=0.5, ls=":")
    ax.plot(x, E_fw[n] + 0.6 * psi_fw[:, n], label=f"n = {n + 1}, E = {E_fw[n]:.3f} eV")
ax.axvspan(-a_well, a_well, color="tab:blue", alpha=0.07)
ax.set_xlim(-1.5, 1.5)
ax.set_xlabel("x (nm)")
ax.set_ylabel("energy (eV)")
ax.set_title("Finite square well: wavefunctions leak into the walls")
ax.legend(fontsize=8, loc="lower right")
plt.show()

Compared with an infinite well of the same width, the wavefunctions don't vanish at the walls; they leak into the classically forbidden region. The highest state, closest to the top, leaks the most. Effectively the well is a bit wider, so the energies are a bit lower than the infinite-well values.

## 8. A potential with no analytic solution

The real power of numerical methods is for potentials nobody can solve by hand. A classic example (Newman, Exercise 6.9) is an infinite square well with a sloped floor,

$$
V(x) = \frac{a\,x}{L} \quad \text{for } 0 < x < L,
$$

with $a = 10$ eV and $L = 0.5$ nm. Physically, this is an electron in a narrow box with an electric field across it, which pushes it toward the left (low-$V$) side. The matrix method handles it with no extra work: just change `V`.

In [ ]:
L_slope, a_slope = 0.5, 10.0                 # nm, eV
N = 1000
h = L_slope / (N + 1)
x = h * np.arange(1, N + 1)
V_slope = a_slope * x / L_slope
E_sl, vecs = np.linalg.eigh(hamiltonian(x, V_slope))
psi_sl = vecs / np.sqrt(h)

E_box = (np.arange(1, 6) * np.pi)**2 * C / L_slope**2
print("sloped well (eV):           ", np.round(E_sl[:5], 3))
print("flat well, same width (eV): ", np.round(E_box, 3))

In [ ]:
fig, ax = plt.subplots()
for n in range(3):
    ax.plot(x, psi_sl[:, n]**2, label=f"n = {n + 1}, E = {E_sl[n]:.2f} eV")
ax.set_xlabel("x (nm)")
ax.set_ylabel(r"probability density $|\psi|^2$ (nm$^{-1}$)")
ax.set_title("Sloped-floor well: the ground state shifts toward low V")
ax.legend()
plt.show()

Higher states are shifted up by close to the average potential, $a/2 = 5$ eV: their kinetic energy is so large that the slope hardly changes their shape. The ground state is shifted by noticeably less (about 4.3 eV), because its probability piles up on the left, where the potential is low.

You could double-check these numbers with the shooting method: only `V` changes. Having two independent methods that agree is one of the best ways to trust a numerical result.

### Your turn 4

A **linear potential** $V(x) = F|x|$ is a toy model for quark confinement (the force between quarks is roughly constant, unlike the $1/r^2$ Coulomb force) and for a particle bouncing on a floor in a uniform field. Use the matrix method with $F = 5$ eV/nm for an electron on a grid from $-3$ to $3$ nm. Find the lowest 6 energies and plot the spacing $E_{n+1} - E_n$ versus $n$. Compare with the harmonic oscillator (constant spacing) and the infinite well (spacing grows with $n$). Plot the first four wavefunctions offset by energy over the potential.

In [ ]:
# Your code here

### Your turn 5

Pick one of the two methods and check its convergence for the **harmonic oscillator**: compute the relative error of $E_0$ and $E_4$ as a function of the grid spacing (matrix method) or of `rtol` and $x_{\max}$ (shooting). For the matrix method, also try making the box smaller ($-1 < x < 1$ nm): which states are affected first, and why?

In [ ]:
# Your code here

## Summary

- The TISE, $-\frac{\hbar^2}{2m}\psi'' + V\psi = E\psi$, has bound-state solutions only at special energies. In eV and nm for an electron, $\hbar^2/2m_e = 0.0381$ eV nm².
- **Shooting method:** treat the TISE as an ODE in $x$ (state vector $(\psi, \psi')$), integrate with `solve_ivp` from one boundary for a trial $E$, and use `brentq` on the mismatch at the other boundary. For symmetric potentials, shoot from $x = 0$ with even ($\psi = 1, \psi' = 0$) or odd ($\psi = 0, \psi' = 1$) starting conditions.
- **Normalize** with `psi / np.sqrt(simpson(psi**2, x=x))`.
- **Linear algebra in NumPy:** `A @ v` (matrix product; `*` is element-wise), `A.T`, `np.eye`, `np.diag(d, k)`, `np.linalg.solve(A, b)`, `np.linalg.eigh(A)` (sorted eigenvalues; eigenvectors are the **columns**), `np.linalg.eigvalsh`.
- **Matrix method:** the finite-difference second derivative turns $\hat H$ into a tridiagonal matrix with $2C/h^2 + V_j$ on the diagonal and $-C/h^2$ next to it. One `eigh` call gives all energies and wavefunctions; normalize with $\psi_j = v_j/\sqrt h$. Error is $O(h^2)$. For large grids use `scipy.linalg.eigh_tridiagonal`.
- Validate against exact results (infinite well $\propto n^2$, oscillator $(n+\frac12)\hbar\omega$, finite-well transcendental roots), check convergence, and compare two independent methods.
- New Python: `enumerate`, conditional expressions, `np.where`, `np.full`, `np.column_stack`, `time.perf_counter`.

## Further reading

- M. Newman, *Computational Physics*, Section 6.1 (linear equations and eigenvalue problems, including Exercise 6.9) and Section 8.6 (the shooting method for the Schrödinger equation).
- NumPy documentation: [linear algebra (`numpy.linalg`)](https://numpy.org/doc/stable/reference/routines.linalg.html).
- SciPy documentation: [`scipy.linalg.eigh_tridiagonal`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.eigh_tridiagonal.html).